# Build a Customer Support Router Agentic RAG System (Modularized)

This notebook is a modularized version of `01_Router_Agentic_RAG_System.ipynb`:

- **Prompts** are loaded from `prompts.yaml` instead of being hardcoded in each node function.
- **Knowledge base retrieval** for all three response nodes is unified into a single `retrieve_knowledge_base_content()` helper.
- **Response generation** for Technical/Billing/General queries is unified into a single `generate_support_response()` implementation (bound to each node via `functools.partial`), backed by one shared `response_generation` prompt template.
- **Query categorization** uses the `ChatPromptTemplate` + LCEL approach (previously called `categorize_inquiry_alt` in the base notebook) as the single implementation.

![](https://i.imgur.com/bLCdxCI.png)

### Intelligent Router Agentic RAG System

This project builds an **Intelligent Router Agentic RAG System** that combines intelligent query analysis, sentiment detection, and dynamic routing with Retrieval-Augmented Generation (RAG) to handle diverse user inquiries efficiently. The workflow includes:

1. **Query Categorization and Sentiment Analysis**:
   - The system uses an LLM to analyze the user's query and determine:
     - **Query Category**: Technical, Billing, or General.
     - **User Sentiment**: Positive, Neutral, or Negative (used to decide whether to escalate).

2. **Intelligent Routing**:
   - Based on **query_category** and **query_sentiment**, the system routes the query to the appropriate handling node:
     - **Escalate to Human**: If the sentiment is negative.
     - **Generate Billing / Technical / General Response**: Otherwise, based on category.

3. **Knowledge Base Integration (RAG)**:
   - Responses are grounded in a **Knowledge Base (Vector Database)**.

4. **Escalation Mechanism**:
   - Negative sentiment triggers escalation to a human agent.

In [ ]:
# !pip install langchain-chroma==0.2.0

## Load Company Knowledge Base

In [ ]:
# or download manually from https://drive.google.com/file/d/1CWHutosAcJ6fiddQW5ogvg7NgLstZJ9j/view?usp=sharing and upload to colab or your notebook location
# !gdown 1CWHutosAcJ6fiddQW5ogvg7NgLstZJ9j

In [ ]:
import json

with open("./docs/router_agent_documents.json", "r") as f:
    knowledge_base = json.load(f)

knowledge_base[:3]

In [ ]:
knowledge_base[-3:]

In [ ]:
from langchain_core.documents import Document
from tqdm import tqdm

processed_docs = []

for doc in tqdm(knowledge_base):
    metadata = doc['metadata']
    data = doc['text']
    processed_docs.append(Document(page_content=data,
                                   metadata=metadata))

processed_docs[:3]

In [ ]:
# ============================================================================
# SETUP: Import LLM Helper Functions
# ============================================================================
# We use helper functions to create LLM instances with proper configuration
# These functions handle API key loading and model configuration

import os
import sys

# Add parent directory to path for importing helpers
sys.path.append(os.path.abspath("../.."))

# Import our LLM factory functions
# - get_groq_llm(): Creates a Groq-hosted LLM (fast inference)
# - get_openai_llm(): Creates an OpenAI GPT model
from helpers.utils import get_groq_llm, get_openai_llm, get_databricks_llm

print("LLM helpers imported successfully!")

# ============================================================================
# CREATE THE LLM AND CHATBOT GRAPH
# ============================================================================

# -----------------------------------------------------------------------------
# Step 1: Initialize the LLM
# We use Groq for fast inference, but you can swap to OpenAI
# -----------------------------------------------------------------------------
llm = get_databricks_llm("databricks-claude-sonnet-4")  # Fast, open-source models hosted by Groq
# Alternative: llm = get_openai_llm()  # OpenAI's GPT models

if hasattr(llm, 'model_name'):
    print(f"LLM initialized: {llm.model_name}")
elif hasattr(llm, 'model'):
    print(f"LLM initialized: {llm.model} (Databricks)")
else:
    print("LLM initialized: Groq LLM")

## Create Vector Database

In [ ]:
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
from langchain_chroma import Chroma

kbase_db = Chroma.from_documents(documents=processed_docs,
                                  collection_name='knowledge_base',
                                  embedding=openai_embed_model,
                                  # need to set the distance function to cosine else it uses euclidean by default
                                  # check https://docs.trychroma.com/guides#changing-the-distance-function
                                  collection_metadata={"hnsw:space": "cosine"},
                                  persist_directory="./knowledge_base")

In [ ]:
kbase_search = kbase_db.as_retriever(search_type="similarity_score_threshold",
                                     search_kwargs={"k": 3, "score_threshold": 0.2})

In [ ]:
query = 'what is your refund policy?'
metadata_filter = {'category' : 'general'}
# Update retriever search_kwargs dynamically
kbase_search.search_kwargs["filter"] = metadata_filter
kbase_search.invoke(query)

In [ ]:
query = 'what is your refund policy'
metadata_filter = {'category' : 'General'}
# Update retriever search_kwargs dynamically
kbase_search.search_kwargs["filter"] = metadata_filter
kbase_search.invoke(query)

In [ ]:
query = 'what is your refund policy'
metadata_filter = {'category' : 'technical'}
# Update retriever search_kwargs dynamically
kbase_search.search_kwargs["filter"] = metadata_filter
kbase_search.invoke(query)

## Define the Customer Inquiry State

We create a `CustomerSupportState` typed dictionary to keep track of each interaction:
- **customer_query**: The text of the customer's question
- **query_category**: Technical, Billing, or General (used for routing)
- **query_sentiment**: Positive, Neutral, or Negative (used for routing)
- **final_response**: The system's response to the customer

In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel

class CustomerSupportState(TypedDict):
    customer_query: str
    query_category: str
    query_sentiment: str
    final_response: str

class QueryCategory(BaseModel):
    categorized_topic: Literal['Technical', 'Billing', 'General']

class QuerySentiment(BaseModel):
    sentiment: Literal['Positive', 'Neutral', 'Negative']

## Create Node Functions

Each function below represents a stage in processing a customer inquiry:

1. **categorize_inquiry**: Classifies the query into Technical, Billing, or General (via `ChatPromptTemplate` + LCEL).
2. **analyze_inquiry_sentiment**: Determines if the sentiment is Positive, Neutral, or Negative.
3. **generate_support_response**: Shared implementation that produces a category-specific response (technical/billing/general), bound via `functools.partial` into `generate_technical_response`, `generate_billing_response`, `generate_general_response`.
4. **escalate_to_human_agent**: Escalates the query to a human if sentiment is negative.
5. **determine_route**: Routes the inquiry to the appropriate response node based on category and sentiment.

In [ ]:
import yaml

with open("./prompts.yaml", "r") as f:
    PROMPTS = yaml.safe_load(f)

print(f"Loaded prompts: {list(PROMPTS.keys())}")

In [ ]:
def retrieve_knowledge_base_content(category: str, query: str) -> str:
    """
    Filter the knowledge base retriever by category and return the retrieved
    documents' content joined into a single string. Shared by all
    generate_*_response nodes to avoid repeating the same filter+retrieve+join logic.
    """
    metadata_filter = {"category": category}
    kbase_search.search_kwargs["filter"] = metadata_filter

    relevant_docs = kbase_search.invoke(query)
    return "\n\n".join(doc.page_content for doc in relevant_docs)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ROUTE_CATEGORY_PROMPT = ChatPromptTemplate.from_template(PROMPTS["route_category"])

def categorize_inquiry(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Classify the customer query into Technical, Billing, or General
    using ChatPromptTemplate + LCEL.
    """

    query = support_state["customer_query"]

    chain = ROUTE_CATEGORY_PROMPT | llm.with_structured_output(QueryCategory)
    route_category = chain.invoke({"customer_query": query})

    return {
        "query_category": route_category.categorized_topic
    }

In [ ]:
categorize_inquiry({"customer_query": "Do you provide pretrained models?"})

In [ ]:
categorize_inquiry({"customer_query": "what is your refund policy?"})

In [ ]:
categorize_inquiry({"customer_query": "what payment methods are accepted?"})

In [ ]:
def analyze_inquiry_sentiment(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Analyze the sentiment of the customer query as Positive, Neutral, or Negative.
    """

    query = support_state["customer_query"]
    prompt = PROMPTS["sentiment_category"].format(customer_query=query)
    sentiment_category = llm.with_structured_output(QuerySentiment).invoke(prompt)

    return {
        "query_sentiment": sentiment_category.sentiment
    }

In [ ]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy?"})

In [ ]:
analyze_inquiry_sentiment({"customer_query": "what is your refund policy? I am really fed up with this product and need to refund it"})

In [ ]:
from functools import partial

def generate_support_response(support_state: CustomerSupportState, category: str) -> CustomerSupportState:
    """
    Provide a category-specific support response (technical/billing/general) by combining
    knowledge base retrieval and the LLM. Shared implementation bound via functools.partial
    into generate_technical_response / generate_billing_response / generate_general_response,
    so the three near-identical node functions no longer need separate copies of this logic.
    """
    categorized_topic = support_state["query_category"]
    query = support_state["customer_query"]

    # Only answer if the routed category matches this node's category
    if categorized_topic.lower() == category:
        retrieved_content = retrieve_knowledge_base_content(category, query)

        prompt = ChatPromptTemplate.from_template(PROMPTS["response_generation"])
        chain = prompt | llm
        reply = chain.invoke({
            "support_type": category,
            "customer_query": query,
            "retrieved_content": retrieved_content
        }).content
    else:
        # For a query routed to a different category, provide a default handling response
        reply = "Apologies I was not able to answer your question, please reach out to +1-xxx-xxxx"

    # Update and return the modified support state
    return {
        "final_response": reply
    }

generate_technical_response = partial(generate_support_response, category="technical")
generate_billing_response = partial(generate_support_response, category="billing")
generate_general_response = partial(generate_support_response, category="general")

In [ ]:
generate_technical_response({"customer_query": "what is your refund policy?", "query_category": "General"})

In [ ]:
generate_technical_response({"customer_query": "do you support on-prem models?", "query_category": "Technical"})

In [ ]:
generate_billing_response({"customer_query": "what payment methods are supported?", "query_category": "Billing"})

In [ ]:
generate_general_response({"customer_query": "what is your refund policy?", "query_category": "General"})

In [ ]:
def escalate_to_human_agent(support_state: CustomerSupportState) -> CustomerSupportState:
    """
    Escalate the query to a human agent if sentiment is negative.
    """

    return {
        "final_response": "Apologies, we are really sorry! Someone from our team will be reaching out to your shortly!"
    }

In [ ]:
def determine_route(support_state: CustomerSupportState) -> str:
    """
    Route the inquiry based on sentiment and category.
    """
    if support_state["query_sentiment"] == "Negative":
        return "escalate_to_human_agent"
    elif support_state["query_category"] == "Technical":
        return "generate_technical_response"
    elif support_state["query_category"] == "Billing":
        return "generate_billing_response"
    else:
        return "generate_general_response"

## Build and Compile the Workflow

We construct a LangGraph workflow with the nodes defined above:
1. **categorize_inquiry** → **analyze_inquiry_sentiment** → **route** to the proper response node.
2. If negative, escalate to a human agent.
3. Otherwise, produce an appropriate response (technical, billing, or general).

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# Create the graph with our typed state
customer_support_graph = StateGraph(CustomerSupportState)

# Add nodes for each function
customer_support_graph.add_node("categorize_inquiry", categorize_inquiry)
customer_support_graph.add_node("analyze_inquiry_sentiment", analyze_inquiry_sentiment)
customer_support_graph.add_node("generate_technical_response", generate_technical_response)
customer_support_graph.add_node("generate_billing_response", generate_billing_response)
customer_support_graph.add_node("generate_general_response", generate_general_response)
customer_support_graph.add_node("escalate_to_human_agent", escalate_to_human_agent)

# Add edges to represent the processing flow
customer_support_graph.add_edge("categorize_inquiry", "analyze_inquiry_sentiment")
customer_support_graph.add_conditional_edges(
    "analyze_inquiry_sentiment",
    determine_route,
    [
        "generate_technical_response",
        "generate_billing_response",
        "generate_general_response",
        "escalate_to_human_agent"
    ]
)

# All terminal nodes lead to the END
customer_support_graph.add_edge("generate_technical_response", END)
customer_support_graph.add_edge("generate_billing_response", END)
customer_support_graph.add_edge("generate_general_response", END)
customer_support_graph.add_edge("escalate_to_human_agent", END)

# Set the entry point for the workflow
customer_support_graph.set_entry_point("categorize_inquiry")

# Compile the graph into a runnable agent
memory = MemorySaver()
compiled_support_agent = customer_support_graph.compile(checkpointer=memory)

## Visualize the Workflow

Below is a generated diagram of the workflow using Mermaid syntax. It shows how each node connects in the graph.

In [ ]:
from IPython.display import display, Image, Markdown

display(Image(compiled_support_agent.get_graph().draw_mermaid_png()))

## Helper Function to Run the Workflow

This function takes a customer query and runs it through our compiled workflow, returning the final results (category, sentiment, and generated response).

In [ ]:
def call_support_agent(agent, prompt, user_session_id, verbose=False):
    events = agent.stream(
        {"customer_query": prompt}, # initial state of the agent
        {"configurable": {"thread_id": user_session_id}},
        stream_mode="values",
    )

    print('Running Agent. Please wait...')
    for event in events:
        if verbose:
                print(event)

    display(Markdown(event['final_response']))

## Testing the Customer Support Workflow

Let's test the workflow with some sample queries to verify categorization, sentiment analysis, and response generation.

In [ ]:
uid = 'jim001'
query = "do you support pre-trained models?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)

In [ ]:
query = "how do I get my invoice?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)

In [ ]:
query = "Can you tell me about your shipping policy?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=False)

In [ ]:
query = "I'm fed up with this faulty hardware, I need a refund"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)

In [ ]:
query = "What are your working hours?"
call_support_agent(agent=compiled_support_agent,
                   prompt=query,
                   user_session_id=uid,
                   verbose=True)